In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../olist_analytics.db")

print("Connected to SQL database successfully.")

Connected to SQL database successfully.


In [2]:
query = """
SELECT 
    ROUND(SUM(total_price), 2) AS total_revenue
FROM orders_analytics;
"""

result = pd.read_sql_query(query, conn)

print(result)

   total_revenue
0     13591643.7


In [3]:
query = """
SELECT
    strftime('%Y-%m', order_purchase_timestamp) AS month,
    ROUND(SUM(total_price), 2) AS monthly_revenue
FROM orders_analytics
GROUP BY month
ORDER BY month;
"""

monthly_revenue = pd.read_sql_query(query, conn)

print(monthly_revenue)

      month  monthly_revenue
0   2016-09           267.36
1   2016-10         49507.66
2   2016-12            10.90
3   2017-01        120312.87
4   2017-02        247303.02
5   2017-03        374344.30
6   2017-04        359927.23
7   2017-05        506071.14
8   2017-06        433038.60
9   2017-07        498031.48
10  2017-08        573971.68
11  2017-09        624401.69
12  2017-10        664219.43
13  2017-11       1010271.37
14  2017-12        743914.17
15  2018-01        950030.36
16  2018-02        844178.71
17  2018-03        983213.44
18  2018-04        996647.75
19  2018-05        996517.68
20  2018-06        865124.31
21  2018-07        895507.22
22  2018-08        854686.33
23  2018-09           145.00
24  2018-10              NaN


In [4]:
query = """
SELECT
    ROUND(AVG(total_price), 2) AS average_order_value
FROM orders_analytics
WHERE total_price IS NOT NULL;
"""

result = pd.read_sql_query(query, conn)

print(result)

   average_order_value
0               137.75


In [6]:
import pandas as pd

products = pd.read_csv("../data/olist_products_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")

products.to_sql(
    "products",
    conn,
    if_exists="replace",
    index=False
)

order_items.to_sql(
    "order_items",
    conn,
    if_exists="replace",
    index=False
)

print("Products and order_items tables loaded successfully.")

Products and order_items tables loaded successfully.


In [7]:
query = """
SELECT
    COALESCE(p.product_category_name, 'Unknown') AS category,
    ROUND(SUM(oi.price), 2) AS revenue
FROM order_items oi
LEFT JOIN products p
    ON oi.product_id = p.product_id
GROUP BY category
ORDER BY revenue DESC;
"""

category_revenue = pd.read_sql_query(query, conn)

print(category_revenue)

                         category     revenue
0                    beleza_saude  1258681.34
1              relogios_presentes  1205005.68
2                 cama_mesa_banho  1036988.68
3                   esporte_lazer   988048.97
4          informatica_acessorios   911954.32
..                            ...         ...
69                         flores     1110.04
70                casa_conforto_2      760.27
71              cds_dvds_musicais      730.00
72  fashion_roupa_infanto_juvenil      569.85
73             seguros_e_servicos      283.29

[74 rows x 2 columns]


In [8]:
query = """
SELECT
    seller_id,
    ROUND(SUM(price), 2) AS revenue,
    COUNT(DISTINCT order_id) AS orders
FROM order_items
GROUP BY seller_id
ORDER BY revenue DESC;
"""

seller_revenue = pd.read_sql_query(query, conn)

print(seller_revenue.head(10))

                          seller_id    revenue  orders
0  4869f7a5dfa277a7dca6462dcf3b52b2  229472.63    1132
1  53243585a1d6dc2643021fd1853d8905  222776.05     358
2  4a3ca9315b744ce9f8e9374361493884  200472.92    1806
3  fa1c13f2614d7b5c4749cbc52fecda94  194042.03     585
4  7c67e1448b00f6e969d365cea6b010ab  187923.89     982
5  7e93a43ef30c4f03f38b393420bc753a  176431.87     336
6  da8622b14eb17ae2831f4ac5b9dab84a  160236.57    1314
7  7a67c85e85bb2ce8582c35f2203ad736  141745.53    1160
8  1025f0e2d44d7041d6cf58b6550e0bfa  138968.55     915
9  955fee9216a65b617aa5c0531780ce60  135171.70    1287


In [9]:
query = """
SELECT
    customer_state AS state,
    ROUND(SUM(total_price), 2) AS revenue,
    COUNT(DISTINCT order_id) AS orders
FROM orders_analytics
WHERE total_price IS NOT NULL
GROUP BY customer_state
ORDER BY revenue DESC;
"""

state_revenue = pd.read_sql_query(query, conn)

print(state_revenue)

   state     revenue  orders
0     SP  5202955.05   41375
1     RJ  1824092.67   12762
2     MG  1585308.03   11544
3     RS   750304.02    5432
4     PR   683083.76    4998
5     SC   520553.34    3612
6     BA   511349.99    3358
7     DF   302603.94    2125
8     GO   294591.95    2007
9     ES   275037.31    2025
10    PE   262788.03    1648
11    CE   227254.71    1327
12    PA   178947.81     970
13    MT   156453.53     903
14    MA   119648.22     740
15    MS   116812.64     709
16    PB   115268.08     532
17    PI    86914.08     493
18    RN    83034.98     482
19    AL    80314.81     411
20    SE    58920.85     345
21    TO    49621.74     279
22    RO    46140.64     247
23    AM    22356.84     147
24    AC    15982.95      81
25    AP    13474.30      68
26    RR     7829.43      46


In [10]:
query = """
SELECT
    ROUND(AVG(review_score), 2) AS average_review_score
FROM orders_analytics
WHERE review_score IS NOT NULL;
"""

result = pd.read_sql_query(query, conn)

print(result)

   average_review_score
0                  4.09


In [12]:
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")

reviews.to_sql(
    "order_reviews",
    conn,
    if_exists="replace",
    index=False
)

print("Reviews table loaded successfully.")

Reviews table loaded successfully.


In [13]:
query = """
SELECT
    COALESCE(p.product_category_name, 'Unknown') AS category,
    ROUND(AVG(r.review_score), 2) AS avg_review_score,
    COUNT(DISTINCT r.order_id) AS reviewed_orders
FROM order_reviews r
JOIN order_items oi
    ON r.order_id = oi.order_id
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY category
HAVING reviewed_orders >= 20
ORDER BY avg_review_score ASC;
"""

category_reviews = pd.read_sql_query(query, conn)

print(category_reviews)

                              category  avg_review_score  reviewed_orders
0                      fraldas_higiene              3.26               27
1                    moveis_escritorio              3.49             1263
2                      casa_conforto_2              3.63               23
3              fashion_roupa_masculina              3.64              111
4                       telefonia_fixa              3.68              214
..                                 ...               ...              ...
63                     livros_tecnicos              4.37              257
64                   livros_importados              4.40               53
65                              flores              4.42               28
66  construcao_ferramentas_ferramentas              4.44               94
67              livros_interesse_geral              4.45              508

[68 rows x 3 columns]


In [14]:
query = """
SELECT
    oi.seller_id,
    ROUND(AVG(r.review_score), 2) AS avg_review_score,
    COUNT(DISTINCT r.order_id) AS reviewed_orders
FROM order_reviews r
JOIN order_items oi
    ON r.order_id = oi.order_id
GROUP BY oi.seller_id
HAVING reviewed_orders >= 20
ORDER BY avg_review_score ASC;
"""

seller_reviews = pd.read_sql_query(query, conn)

print(seller_reviews)

                            seller_id  avg_review_score  reviewed_orders
0    ffff564a4f9085cd26170f4732393726              2.10               20
1    1ca7077d890b907f89be8c954a02686a              2.20              114
2    2709af9587499e95e803a6498a5a56e9              2.57               25
3    2eb70248d66e0e3ef83659f71b244378              2.71              198
4    b19f3ca2ea475913750f25a5c37c8d8f              2.79               24
..                                ...               ...              ...
806  d13e50eaa47b4cbe9eb81465865d8cfc              4.81               67
807  d9bd94811c3338dceb4181f3dbc0c73e              4.82               54
808  02f5837340d7eb4f653d676c7256523a              4.83               30
809  41c2bad7229b0c25e6becf179ebf63ff              4.96               20
810  48efc9d94a9834137efd9ea76b065a38              5.00               33

[811 rows x 3 columns]


In [15]:
query = """
SELECT
    customer_state AS state,
    ROUND(AVG(review_score), 2) AS avg_review_score,
    COUNT(DISTINCT order_id) AS reviewed_orders
FROM orders_analytics
WHERE review_score IS NOT NULL
GROUP BY customer_state
ORDER BY avg_review_score ASC;
"""

state_reviews = pd.read_sql_query(query, conn)

print(state_reviews)

   state  avg_review_score  reviewed_orders
0     RR              3.61               46
1     AL              3.76              410
2     MA              3.76              742
3     SE              3.81              349
4     CE              3.85             1326
5     PA              3.85              962
6     BA              3.86             3340
7     RJ              3.88            12687
8     PI              3.92              490
9     PE              4.01             1635
10    PB              4.02              530
11    ES              4.04             2006
12    GO              4.04             2007
13    AC              4.05               81
14    RO              4.05              252
15    DF              4.07             2128
16    SC              4.07             3609
17    MT              4.10              900
18    TO              4.10              279
19    MS              4.11              713
20    RN              4.11              480
21    RS              4.13      

In [16]:
query = """
SELECT
    CASE
        WHEN total_price < 50 THEN 'Under 50'
        WHEN total_price < 100 THEN '50-100'
        WHEN total_price < 250 THEN '100-250'
        WHEN total_price < 500 THEN '250-500'
        ELSE '500+'
    END AS order_value_range,
    ROUND(AVG(review_score), 2) AS avg_review_score,
    COUNT(DISTINCT order_id) AS reviewed_orders
FROM orders_analytics
WHERE total_price IS NOT NULL
  AND review_score IS NOT NULL
GROUP BY order_value_range
ORDER BY
    CASE order_value_range
        WHEN 'Under 50' THEN 1
        WHEN '50-100' THEN 2
        WHEN '100-250' THEN 3
        WHEN '250-500' THEN 4
        WHEN '500+' THEN 5
    END;
"""

order_value_reviews = pd.read_sql_query(query, conn)

print(order_value_reviews)

  order_value_range  avg_review_score  reviewed_orders
0          Under 50              4.17            29198
1            50-100              4.12            28107
2           100-250              4.08            29776
3           250-500              3.97             7239
4              500+              3.94             3597


In [17]:
query = """
SELECT
    ROUND(AVG(
        julianday(order_delivered_customer_date)
        - julianday(order_purchase_timestamp)
    ), 2) AS avg_actual_delivery_days,

    ROUND(AVG(
        julianday(order_estimated_delivery_date)
        - julianday(order_purchase_timestamp)
    ), 2) AS avg_estimated_delivery_days
FROM orders_analytics
WHERE order_delivered_customer_date IS NOT NULL
  AND order_purchase_timestamp IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL;
"""

result = pd.read_sql_query(query, conn)

print(result)

   avg_actual_delivery_days  avg_estimated_delivery_days
0                     12.56                        23.74


In [18]:
query = """
SELECT
    ROUND(AVG(
        julianday(order_delivered_customer_date)
        - julianday(order_estimated_delivery_date)
    ), 2) AS avg_delivery_delay_days
FROM orders_analytics
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL;
"""

result = pd.read_sql_query(query, conn)

print(result)

   avg_delivery_delay_days
0                   -11.18


In [19]:
query = """
SELECT
    COUNT(*) AS delivered_orders,
    SUM(
        CASE
            WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN order_delivered_customer_date > order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS late_delivery_percentage
FROM orders_analytics
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL;
"""

result = pd.read_sql_query(query, conn)

print(result)

   delivered_orders  late_orders  late_delivery_percentage
0             96476         7827                      8.11


In [20]:
query = """
SELECT
    oi.seller_id,
    COUNT(DISTINCT oi.order_id) AS delivered_orders,
    COUNT(DISTINCT CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
        THEN oi.order_id
    END) AS late_orders,
    ROUND(
        100.0 * COUNT(DISTINCT CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN oi.order_id
        END) / COUNT(DISTINCT oi.order_id),
        2
    ) AS late_delivery_percentage
FROM order_items oi
JOIN orders_analytics o
    ON oi.order_id = o.order_id
WHERE o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY oi.seller_id
HAVING delivered_orders >= 20
ORDER BY late_delivery_percentage DESC;
"""

seller_delivery = pd.read_sql_query(query, conn)

print(seller_delivery)

                            seller_id  delivered_orders  late_orders  \
0    f76a3b1349b6df1ee875d1f3fa4340f0                24            9   
1    821fb029fc6e495ca4f08a35d51e53a5                24            9   
2    ede0c03645598cdfc63ca8237acbe73d                43           15   
3    ad781527c93d00d89a11eecd9dcad7c1                38           12   
4    835f0f7810c76831d6c7d24c7a646d4d                42           13   
..                                ...               ...          ...   
799  272f092de69afedd4d2969440b37f18f                25            0   
800  14a08204d03bb6b6bde8029f801ae0eb                28            0   
801  0ebd97a106433a45a4aebe57c1799778                31            0   
802  02ecc2a19303f05e59ce133fd923fff7                21            0   
803  013900e863eace745d3ec7614cab5b1a                23            0   

     late_delivery_percentage  
0                       37.50  
1                       37.50  
2                       34.88  
3      